In [0]:
# ==========================================================
# LAB 02 - Retail Customer 360
# Customer Data Ingestion - GitHub to Delta Lake
# ==========================================================

CATALOG = "retail_customer360"
BRONZE_SCHEMA = "bronze"

SOURCE_URL = (
    "https://raw.githubusercontent.com/"
    "normanmartinez2/retail_customer360/"
    "refs/heads/main/data/crm/customers_lab02_v1.csv"
)

TABLE_NAME = f"{CATALOG}.{BRONZE_SCHEMA}.customers_raw"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print("Catalog:", spark.catalog.currentCatalog())
print("Schema:", spark.catalog.currentDatabase())
print("Target table:", TABLE_NAME)
print("Source:", SOURCE_URL)

Catalog: retail_customer360
Schema: bronze
Target table: retail_customer360.bronze.customers_raw
Source: https://raw.githubusercontent.com/normanmartinez2/retail_customer360/refs/heads/main/data/crm/customers_lab02_v1.csv


In [0]:
# Create landing volume

VOLUME_NAME = "crm_source_files"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}
    COMMENT 'Landing zone for CRM source files'
""")

print("Volume ready:", VOLUME_NAME)

Volume ready: crm_source_files


In [0]:
# Define landing file path

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{BRONZE_SCHEMA}/{VOLUME_NAME}"
)

CSV_PATH = f"{VOLUME_PATH}/customers_lab02_v1.csv"

print("Landing path:", CSV_PATH)

Landing path: /Volumes/retail_customer360/bronze/crm_source_files/customers_lab02_v1.csv


In [0]:
# ==========================================================
# Download source dataset from GitHub
# ==========================================================

import urllib.request

try:
    with urllib.request.urlopen(
        SOURCE_URL,
        timeout=30
    ) as response:

        csv_bytes = response.read()

    csv_content = csv_bytes.decode("utf-8-sig")

    print("Download successful")
    print("File size:", len(csv_bytes), "bytes")

except Exception as e:
    print("Download failed")
    print("Error:", str(e))

    raise

Download successful
File size: 220151 bytes


In [0]:
# Store source file in Unity Catalog Volume

dbutils.fs.put(
    CSV_PATH,
    csv_content,
    overwrite=True
)

print("File stored successfully")
print("Path:", CSV_PATH)

Wrote 220151 bytes.
File stored successfully
Path: /Volumes/retail_customer360/bronze/crm_source_files/customers_lab02_v1.csv


In [0]:
# List files in landing volume

display(
    dbutils.fs.ls(VOLUME_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/retail_customer360/bronze/crm_source_files/customers_lab02_v1.csv,customers_lab02_v1.csv,220151,1791626882000


In [0]:
# ==========================================================
# Define CRM Customer Schema
# ==========================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

customer_schema = StructType([

    StructField("customer_id", StringType(), True),

    StructField("first_name", StringType(), True),

    StructField("last_name", StringType(), True),

    StructField("email", StringType(), True),

    StructField("phone", StringType(), True),

    StructField("date_of_birth", StringType(), True),

    StructField("country", StringType(), True),

    StructField("city", StringType(), True),

    StructField("postal_code", StringType(), True),

    StructField("customer_segment", StringType(), True),

    StructField("marketing_consent", StringType(), True),

    StructField("created_at", StringType(), True),

    StructField("updated_at", StringType(), True),

    StructField("source_system", StringType(), True),

    StructField("notes", StringType(), True)

])

print("CRM schema defined successfully")

CRM schema defined successfully


In [0]:
# ==========================================================
# Read CRM Customer CSV using PySpark
# ==========================================================

df_customers = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(customer_schema)
    .load(CSV_PATH)
)

display(df_customers.limit(20))

customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes
null,Lucía,Apellido0001,customer0001@example.com,+34910000001,1970-01-12,ES,Madrid,28001,Silver,false,2024-01-02T09:00:00Z,2024-02-01T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Mateo,Apellido0002,customer0002@example.com,+34910000002,1970-01-23,ES,Barcelona,08001,Gold,true,2024-01-03T09:00:00Z,2024-02-02T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Zoë,Apellido0003,customer0003@example.com,+34910000003,1970-02-03,FR,Paris,75001,Standard,false,2024-01-04T09:00:00Z,2024-02-03T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,André,Apellido0004,customer0004@example.com,+34910000004,1970-02-14,PT,Lisboa,01000,Silver,true,2024-01-05T09:00:00Z,2024-02-04T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Sofía,Apellido0005,customer0005@example.com,+34910000005,1970-02-25,DE,Berlin,10115,Gold,false,2024-01-06T09:00:00Z,2024-02-05T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Łukasz,Apellido0006,customer0006@example.com,+34910000006,1970-03-08,ES,Madrid,28001,Standard,true,2024-01-07T09:00:00Z,2024-02-06T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Marta,Apellido0007,customer0007@example.com,+34910000007,1970-03-19,ES,Barcelona,08001,Silver,false,2024-01-08T09:00:00Z,2024-02-07T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Óscar,Apellido0008,customer0008@example.com,+34910000008,1970-03-30,FR,Paris,75001,Gold,true,2024-01-09T09:00:00Z,2024-02-08T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Chloé,Apellido0009,customer0009@example.com,+34910000009,1970-04-10,PT,Lisboa,01000,Standard,false,2024-01-10T09:00:00Z,2024-02-09T10:00:00Z,CRM_SIMULATED,Cliente sintético
null,Núria,Apellido0010,customer0010@example.com,+34910000010,1970-04-21,DE,Berlin,10115,Silver,true,2024-01-11T09:00:00Z,2024-02-10T10:00:00Z,CRM_SIMULATED,Cliente sintético


In [0]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- marketing_consent: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- notes: string (nullable = true)



In [0]:
# Count incoming records

total_records = df_customers.count()

print("Total CRM records:", total_records)

Total CRM records: 1223


In [0]:
# Validate expected schema

expected_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "date_of_birth",
    "country",
    "city",
    "postal_code",
    "customer_segment",
    "marketing_consent",
    "created_at",
    "updated_at",
    "source_system",
    "notes"
]

assert df_customers.columns == expected_columns, (
    "Unexpected CRM schema"
)

assert total_records >= 1000, (
    "Dataset contains fewer than 1000 records"
)

print("Source dataset validation passed")

Source dataset validation passed


In [0]:
# ==========================================================
# Add ingestion metadata
# ==========================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

BATCH_ID = "LAB02_BATCH_001"

df_bronze = (
    df_customers

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

    .withColumn(
        "ingestion_batch_id",
        lit(BATCH_ID)
    )

    .withColumn(
        "source_file",
        lit("customers_lab02_v1.csv")
    )

    .withColumn(
        "source_repository",
        lit(
            "github.com/normanmartinez2/"
            "retail_customer360"
        )
    )

    .withColumn(
        "ingestion_layer",
        lit("BRONZE")
    )
)

display(df_bronze.limit(20))

customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes,ingestion_timestamp,ingestion_batch_id,source_file,source_repository,ingestion_layer
null,Lucía,Apellido0001,customer0001@example.com,+34910000001,1970-01-12,ES,Madrid,28001,Silver,false,2024-01-02T09:00:00Z,2024-02-01T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Mateo,Apellido0002,customer0002@example.com,+34910000002,1970-01-23,ES,Barcelona,08001,Gold,true,2024-01-03T09:00:00Z,2024-02-02T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Zoë,Apellido0003,customer0003@example.com,+34910000003,1970-02-03,FR,Paris,75001,Standard,false,2024-01-04T09:00:00Z,2024-02-03T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,André,Apellido0004,customer0004@example.com,+34910000004,1970-02-14,PT,Lisboa,01000,Silver,true,2024-01-05T09:00:00Z,2024-02-04T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Sofía,Apellido0005,customer0005@example.com,+34910000005,1970-02-25,DE,Berlin,10115,Gold,false,2024-01-06T09:00:00Z,2024-02-05T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Łukasz,Apellido0006,customer0006@example.com,+34910000006,1970-03-08,ES,Madrid,28001,Standard,true,2024-01-07T09:00:00Z,2024-02-06T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Marta,Apellido0007,customer0007@example.com,+34910000007,1970-03-19,ES,Barcelona,08001,Silver,false,2024-01-08T09:00:00Z,2024-02-07T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Óscar,Apellido0008,customer0008@example.com,+34910000008,1970-03-30,FR,Paris,75001,Gold,true,2024-01-09T09:00:00Z,2024-02-08T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Chloé,Apellido0009,customer0009@example.com,+34910000009,1970-04-10,PT,Lisboa,01000,Standard,false,2024-01-10T09:00:00Z,2024-02-09T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Núria,Apellido0010,customer0010@example.com,+34910000010,1970-04-21,DE,Berlin,10115,Silver,true,2024-01-11T09:00:00Z,2024-02-10T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:12:07.783Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE


In [0]:
# ==========================================================
# Write CRM data into Bronze Delta Table
# ==========================================================

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLE_NAME)
)

print("Bronze Delta table created successfully")
print("Table:", TABLE_NAME)

Bronze Delta table created successfully
Table: retail_customer360.bronze.customers_raw


In [0]:
# Read Bronze Delta table

df_result = spark.sql(f"""
    SELECT *
    FROM {TABLE_NAME}
""")

display(df_result.limit(20))

customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes,ingestion_timestamp,ingestion_batch_id,source_file,source_repository,ingestion_layer
null,Lucía,Apellido0001,customer0001@example.com,+34910000001,1970-01-12,ES,Madrid,28001,Silver,false,2024-01-02T09:00:00Z,2024-02-01T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Mateo,Apellido0002,customer0002@example.com,+34910000002,1970-01-23,ES,Barcelona,08001,Gold,true,2024-01-03T09:00:00Z,2024-02-02T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Zoë,Apellido0003,customer0003@example.com,+34910000003,1970-02-03,FR,Paris,75001,Standard,false,2024-01-04T09:00:00Z,2024-02-03T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,André,Apellido0004,customer0004@example.com,+34910000004,1970-02-14,PT,Lisboa,01000,Silver,true,2024-01-05T09:00:00Z,2024-02-04T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Sofía,Apellido0005,customer0005@example.com,+34910000005,1970-02-25,DE,Berlin,10115,Gold,false,2024-01-06T09:00:00Z,2024-02-05T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Łukasz,Apellido0006,customer0006@example.com,+34910000006,1970-03-08,ES,Madrid,28001,Standard,true,2024-01-07T09:00:00Z,2024-02-06T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Marta,Apellido0007,customer0007@example.com,+34910000007,1970-03-19,ES,Barcelona,08001,Silver,false,2024-01-08T09:00:00Z,2024-02-07T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Óscar,Apellido0008,customer0008@example.com,+34910000008,1970-03-30,FR,Paris,75001,Gold,true,2024-01-09T09:00:00Z,2024-02-08T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Chloé,Apellido0009,customer0009@example.com,+34910000009,1970-04-10,PT,Lisboa,01000,Standard,false,2024-01-10T09:00:00Z,2024-02-09T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Núria,Apellido0010,customer0010@example.com,+34910000010,1970-04-21,DE,Berlin,10115,Silver,true,2024-01-11T09:00:00Z,2024-02-10T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE


In [0]:
# Count customers by country

display(
    spark.sql(f"""
        SELECT
            country,
            COUNT(*) AS total_records
        FROM {TABLE_NAME}
        GROUP BY country
        ORDER BY total_records DESC
    """)
)

country,total_records
ES,482
FR,242
DE,242
PT,242
es,7
XX,5
null,3


In [0]:
# Inspect Delta table metadata

display(
    spark.sql(f"""
        DESCRIBE DETAIL {TABLE_NAME}
    """)
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,696503b5-f265-47ab-9f8f-851c0fde5024,retail_customer360.bronze.customers_raw,null,,2026-10-10T10:13:39.589Z,2026-10-10T10:13:44.000Z,List(),List(),1,16205,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 5a1ae5fb-618e-42f2-b015-e8659ebc1ac9)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
# Explore Delta transaction history

display(
    spark.sql(f"""
        DESCRIBE HISTORY {TABLE_NAME}
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-10T10:13:44.000Z,72581255045954,norman.martinez2@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""5a1ae5fb-618e-42f2-b015-e8659ebc1ac9""}, statsOnLoad -> true)",null,List(3478402565557876),11d09708-9835-4b9b-bbec-2f3deab1056e,1010-090107-irsb8gvm-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1223, numOutputBytes -> 16205)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
from pyspark.sql.functions import (
    col,
    trim
)

# Find customers with missing IDs

df_missing_ids = df_result.filter(
    col("customer_id").isNull() |
    (trim(col("customer_id")) == "")
)

display(df_missing_ids)

print(
    "Records with missing customer_id:",
    df_missing_ids.count()
)

customer_id,first_name,last_name,email,phone,date_of_birth,country,city,postal_code,customer_segment,marketing_consent,created_at,updated_at,source_system,notes,ingestion_timestamp,ingestion_batch_id,source_file,source_repository,ingestion_layer
null,Lucía,Apellido0001,customer0001@example.com,+34910000001,1970-01-12,ES,Madrid,28001,Silver,false,2024-01-02T09:00:00Z,2024-02-01T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Mateo,Apellido0002,customer0002@example.com,+34910000002,1970-01-23,ES,Barcelona,08001,Gold,true,2024-01-03T09:00:00Z,2024-02-02T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Zoë,Apellido0003,customer0003@example.com,+34910000003,1970-02-03,FR,Paris,75001,Standard,false,2024-01-04T09:00:00Z,2024-02-03T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,André,Apellido0004,customer0004@example.com,+34910000004,1970-02-14,PT,Lisboa,01000,Silver,true,2024-01-05T09:00:00Z,2024-02-04T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Sofía,Apellido0005,customer0005@example.com,+34910000005,1970-02-25,DE,Berlin,10115,Gold,false,2024-01-06T09:00:00Z,2024-02-05T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Łukasz,Apellido0006,customer0006@example.com,+34910000006,1970-03-08,ES,Madrid,28001,Standard,true,2024-01-07T09:00:00Z,2024-02-06T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Marta,Apellido0007,customer0007@example.com,+34910000007,1970-03-19,ES,Barcelona,08001,Silver,false,2024-01-08T09:00:00Z,2024-02-07T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Óscar,Apellido0008,customer0008@example.com,+34910000008,1970-03-30,FR,Paris,75001,Gold,true,2024-01-09T09:00:00Z,2024-02-08T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Chloé,Apellido0009,customer0009@example.com,+34910000009,1970-04-10,PT,Lisboa,01000,Standard,false,2024-01-10T09:00:00Z,2024-02-09T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE
null,Núria,Apellido0010,customer0010@example.com,+34910000010,1970-04-21,DE,Berlin,10115,Silver,true,2024-01-11T09:00:00Z,2024-02-10T10:00:00Z,CRM_SIMULATED,Cliente sintético,2026-10-10T10:13:40.584Z,LAB02_BATCH_001,customers_lab02_v1.csv,github.com/normanmartinez2/retail_customer360,BRONZE


Records with missing customer_id: 10


In [0]:
# Find duplicate customer identifiers

display(
    spark.sql(f"""
        SELECT
            customer_id,
            COUNT(*) AS occurrences
        FROM {TABLE_NAME}
        WHERE customer_id IS NOT NULL
          AND TRIM(customer_id) <> ''
        GROUP BY customer_id
        HAVING COUNT(*) > 1
        ORDER BY occurrences DESC
    """)
)

customer_id,occurrences
"Segunda línea: envío a domicilio""",3
C000302,2
C000208,2
C000307,2
C000209,2
C000301,2
C000305,2
C000304,2
C000306,2
C000206,2


In [0]:
# Find missing emails

display(
    spark.sql(f"""
        SELECT
            customer_id,
            first_name,
            last_name,
            email
        FROM {TABLE_NAME}
        WHERE email IS NULL
           OR TRIM(email) = ''
    """)
)

customer_id,first_name,last_name,email
C000011,Lucía,Apellido0011,null
C000012,Mateo,Apellido0012,null
C000013,Zoë,Apellido0013,null
C000014,André,Apellido0014,null
C000015,Sofía,Apellido0015,null
C000016,Łukasz,Apellido0016,null
C000017,Marta,Apellido0017,null
C000018,Óscar,Apellido0018,null
C000019,Chloé,Apellido0019,null
C000020,Núria,Apellido0020,null


In [0]:
# Find invalid email formats

from pyspark.sql.functions import lower

email_pattern = (
    r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
)

df_invalid_emails = df_result.filter(
    col("email").isNotNull() &
    (trim(col("email")) != "") &
    ~lower(trim(col("email"))).rlike(email_pattern)
)

display(df_invalid_emails.select(
    "customer_id",
    "first_name",
    "email"
))

print(
    "Invalid emails:",
    df_invalid_emails.count()
)

customer_id,first_name,email
C000026,Łukasz,correo-sin-arroba
C000027,Marta,correo-sin-arroba
C000028,Óscar,correo-sin-arroba
C000029,Chloé,correo-sin-arroba
C000030,Núria,correo-sin-arroba
C000031,Lucía,correo-sin-arroba
C000032,Mateo,correo-sin-arroba
C000033,Zoë,correo-sin-arroba
C000034,André,correo-sin-arroba
C000035,Sofía,correo-sin-arroba


Invalid emails: 12


In [0]:
# Find malformed birth dates

display(
    spark.sql(f"""
        SELECT
            customer_id,
            first_name,
            date_of_birth
        FROM {TABLE_NAME}
        WHERE date_of_birth IS NOT NULL
          AND TRY_CAST(date_of_birth AS DATE) IS NULL
    """)
)

customer_id,first_name,date_of_birth
C000056,Łukasz,1980-02-30
C000057,Marta,1980-02-30
C000058,Óscar,1980-02-30
C000059,Chloé,1980-02-30
C000060,Núria,1980-02-30
C000061,Lucía,1980-02-30


In [0]:
# Find future birth dates

display(
    spark.sql(f"""
        SELECT
            customer_id,
            first_name,
            date_of_birth
        FROM {TABLE_NAME}
        WHERE TRY_CAST(date_of_birth AS DATE)
              > CURRENT_DATE()
    """)
)

customer_id,first_name,date_of_birth
C000062,Mateo,2099-01-01
C000063,Zoë,2099-01-01
C000064,André,2099-01-01
C000065,Sofía,2099-01-01


In [0]:
# Explore country codes

display(
    spark.sql(f"""
        SELECT
            country,
            COUNT(*) AS total_records
        FROM {TABLE_NAME}
        GROUP BY country
        ORDER BY total_records DESC
    """)
)

country,total_records
ES,482
FR,242
DE,242
PT,242
es,7
XX,5
null,3


In [0]:
# ==========================================================
# CRM DATA QUALITY REPORT
# ==========================================================

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    expr,
    current_date,
    count,
    when,
    sum as spark_sum
)

email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

df_quality = df_result.select(

    "*",

    when(
        col("customer_id").isNull() |
        (trim(col("customer_id")) == ""),
        1
    ).otherwise(0).alias("missing_customer_id"),

    when(
        col("email").isNull() |
        (trim(col("email")) == ""),
        1
    ).otherwise(0).alias("missing_email"),

    when(
        col("email").isNotNull() &
        (trim(col("email")) != "") &
        ~lower(trim(col("email"))).rlike(email_pattern),
        1
    ).otherwise(0).alias("invalid_email"),

    when(
        col("date_of_birth").isNotNull() &
        expr(
            "try_cast(date_of_birth as date) IS NULL"
        ),
        1
    ).otherwise(0).alias("invalid_birth_date"),

    when(
        expr(
            "try_cast(date_of_birth as date)"
        ) > current_date(),
        1
    ).otherwise(0).alias("future_birth_date")

)

quality_report = df_quality.agg(

    count("*").alias("total_records"),

    spark_sum(
        "missing_customer_id"
    ).alias("missing_ids"),

    spark_sum(
        "missing_email"
    ).alias("missing_emails"),

    spark_sum(
        "invalid_email"
    ).alias("invalid_emails"),

    spark_sum(
        "invalid_birth_date"
    ).alias("invalid_birth_dates"),

    spark_sum(
        "future_birth_date"
    ).alias("future_birth_dates")

)

display(quality_report)


total_records,missing_ids,missing_emails,invalid_emails,invalid_birth_dates,future_birth_dates
1223,10,18,12,6,4


In [0]:
# Show tables in Bronze

display(
    spark.sql("""
        SHOW TABLES
        IN retail_customer360.bronze
    """)
)

database,tableName,isTemporary
bronze,customers_raw,false


In [0]:
# Show volumes in Bronze

display(
    spark.sql("""
        SHOW VOLUMES
        IN retail_customer360.bronze
    """)
)

database,volume_name
bronze,crm_source_files


In [0]:
# ==========================================================
# LAB 02 - FINAL VALIDATION
# ==========================================================

TABLE_NAME = (
    "retail_customer360.bronze.customers_raw"
)

df_final = spark.table(TABLE_NAME)

total_records = df_final.count()
total_columns = len(df_final.columns)

table_format = spark.sql(f"""
    DESCRIBE DETAIL {TABLE_NAME}
""").select("format").first()[0]

print("=" * 60)
print("RETAIL CUSTOMER 360 - LAB 02")
print("=" * 60)

print("Source: GitHub CRM Dataset")
print("Target:", TABLE_NAME)

print("Total records:", total_records)
print("Total columns:", total_columns)
print("Table format:", table_format)

assert total_records >= 1000, (
    "Unexpected record count"
)

assert total_columns == 20, (
    "Unexpected column count"
)

assert table_format.lower() == "delta", (
    "Target table is not Delta"
)

print("=" * 60)
print("LAB 02 VALIDATION PASSED")
print("=" * 60)

RETAIL CUSTOMER 360 - LAB 02
Source: GitHub CRM Dataset
Target: retail_customer360.bronze.customers_raw
Total records: 1223
Total columns: 20
Table format: delta
LAB 02 VALIDATION PASSED
